In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Importing the necessary libraries for reading and processing the data**

In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt

The code begins by importing the required libraries, including numpy, pandas, tensorflow, and necessary modules from tensorflow.keras, as well as modules from sklearn for data preprocessing and evaluation.

**Reading the data from the file**
We create a variable **path** which stores the path of the input dataset/file and using the **read_csv** function of **pandas** library read the dataset and store it in another variable **data**.

In [ ]:
path = '/kaggle/input/fraud-detection-credit-card/creditcard.csv'
data = pandas.read_csv(path)

In [ ]:
data.head(10)

In [ ]:
data.tail(10)

We count the values of  the **class** column which are **0 and 1** and generate a distribution of these values in the data. We then plot the distribution into a **bar graph** and a **pie chart**.

In [ ]:
a = data['class'].value_counts().rename('count')
b = (data['class'].value_counts(normalize=True)*100).rename('distribution')

t = pandas.concat([a,b], axis=1)
t.index = ["Genuine", "Fraud"]
t['distribution'].plot(kind='bar', figsize=[5,5])
t

In [ ]:
X = data.drop('class', axis=1)
y = data['class']

In [ ]:
t['distribution'].plot(kind='pie', figsize=[4,4])

In [ ]:
data.isnull().sum()

In [ ]:
data.dtypes

**Importing the necessary libraries for the ML Model**

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix as cm
from sklearn.model_selection import train_test_split
import seaborn as sns

**Split the data into training and testing data**

For this we use the **train_test_split** function of the **sklearn.model_selection** library. We can decide the test size by giving it any number between 0 and 1 as a parameter. The **test_size** variable then divides the data by the same percentage as the number given to it as parameter.

**xtrain** and **xtest** contain the **input features**, while **ytrain** and **ytest** contain the corresponding **target labels**.

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(data.iloc[:,:-1], data.iloc[:, -1], test_size=0.3, random_state=60)

In [ ]:
train_fraud = xtrain[ytrain == 1]
train_genuine = xtrain[ytrain == 0]

train_fraud_count = len(train_fraud)
train_genuine_count = len(train_genuine)

plt.figure(figsize=(8, 12))
plt.bar(['Fraud', 'Genuine'], [train_fraud_count, train_genuine_count], color=['red', 'blue'])
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Distribution of Fraud and Genuine Transactions (Training Set)')
plt.show()

In [ ]:
test_fraud = xtest[ytest == 1]
test_genuine = xtest[ytest == 0]

test_fraud_count = len(test_fraud)
test_genuine_count = len(test_genuine)

plt.figure(figsize=(8, 12))
plt.bar(['Fraud', 'Genuine'], [test_fraud_count, test_genuine_count], color=['red', 'blue'])
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Distribution of Fraud and Genuine Transactions (Test Set)')
plt.show()

In [ ]:
print("Fraud cases in training data : ", train_fraud_count)
print("Genuine Cases cases in training data : ", train_genuine_count)
print("Fraud cases in test data : ", test_fraud_count)
print("Fraud cases in test data : ", test_genuine_count)

The data is standardized using **StandardScaler** from **sklearn.preprocessing**. It **scales the features to have zero mean and unit variance**. The **fit_transform** method is applied to the **training data**, and the **transform** method is used for the **testing data**. The scaler is fitted on the training data and applied to both sets.

In [ ]:
scaler = StandardScaler()
xtrain = scaler.fit_transform(xtrain)
xtest = scaler.transform(xtest)

Create the Deep Learning Model which has to be trained for the data

The deep learning model is constructed using the **Sequential** API from **tensorflow.keras**. The model architecture consists of several fully connected dense layers.

The first layer (**Dense**) has **16 units/neurons**, equal to the number of input features (**xtrain.shape[1]**). The activation function used is **ReLU ('relu')**, which introduces non-linearity.

The next layer (**Dense**) has **24 units** with ReLU activation.

A **Dropout layer with a dropout rate of 0.5 is added to mitigate overfitting**.

Another **Dense layer with 20 units and ReLU activation** follows.

The **final output layer (Dense)** has a **single unit** with **sigmoid activation**. Since it's a **binary classification task (fraud or not fraud)**, the **sigmoid activation function squeezes the output between 0 and 1**, representing the probability of fraud.

The **model.summary()** function provides a **summary of the model's architecture**, showing the number of parameters and the flow of data through the layers.

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(units=16, input_dim=xtrain.shape[1], activation='relu'),
    keras.layers.Dense(units=24, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=20, activation='relu'),
    keras.layers.Dense(units=1, activation='sigmoid')
])
model.summary()

Fit the Training data in the Deep Learning Model to train the model

The model is compiled with the **Adam optimizer ('adam')** and **binary cross-entropy loss function ('binary_crossentropy')** suitable for binary classification. The metric used for evaluation is **accuracy**.

Training is performed using the **fit** method, specifying the training data (xtrain and ytrain), batch size, and the number of epochs.

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(xtrain, ytrain, batch_size=15, epochs=5)

Predict the outcome from the model via the test data

The model is evaluated on the test data using the **predict** method. Predictions are **obtained as probabilities** (ypred).

A **threshold of 0.4** is applied to convert the probabilities into class labels. **Values above the threshold are classified as fraud (1), and values below or equal to the threshold are classified as genuine (0)**.

The **confusion matrix** (cmat) is computed using **confusion_matrix** from **sklearn.metrics**. It **provides a breakdown of the predicted labels compared to the true labels**.

The confusion matrix is printed to display the results of the model's performance.

In [ ]:
ypred = model.predict(xtest)
ypred = (ypred > 0.4).astype(int)
cmat = cm(ytest, ypred)
print(cmat)

We then use the formula:

**accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)**

to calculate the accuracy of our model.

In [ ]:
true_positives = cmat[1][1]
true_negatives = cmat[0][0]
false_positives = cmat[0][1]
false_negatives = cmat[1][0]

accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
print("Accuracy:", accuracy)

save the Deep Learning Model

In [ ]:
model.save("ccfd.h5")

Load the Saved Model

In [ ]:
from tensorflow.keras.models import load_model

loaded_model = load_model("ccfd.h5")

We Can now input new data under the new_data variable and pass it to the model for evaluation/classification into fraud or genuine.

In [ ]:
new_data = scaler.transform(xtest)
predictions = loaded_model.predict(new_data)
class_labels = (predictions > 0.4).astype(int)
cmat2 = cm(ytest, class_labels)

true_positives = cmat2[1][1]
true_negatives = cmat2[0][0]
false_positives = cmat2[0][1]
false_negatives = cmat2[1][0]

accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
print("Accuracy:", accuracy)

In [ ]:
print(accuracy)

In [ ]:
print(class_labels)

Get to know if the transactions as given in data are genuine or fraud. If the probability is greater than 0.5 it marks the transaction as fraudulent and stores the result in class_labels variable. From the class_labels we print which data is fraud and which is genuine.

In [ ]:
gen = 0
fraud = 0
for label in class_labels:
    if label == 0:
        print("Transaction is genuine")
        gen+=1
    else:
        print("Transaction is fraudulent")
        fraud+=1

Counting the number of fraudaulent and genuine transactions. Used classical Python method just to keep the code simple so that reader is able to understand properly.

In [ ]:
print(gen)
print(fraud)

We can easily see that the number of genuine cases are very high in comparision to the number of fraudaulent cases (which are only 60). This is mainly due to the fact that the dataset contains majorly genuine cases which account to approx 90% of the total data.

Gives the probability of a transaction being fraudaulent for each data in the dataset.

In [ ]:
prob = (fraud/(gen+fraud))
prob

In [ ]:
t['distribution']

In [ ]:
for probability in predictions:
    print(probability)